In [125]:
import re
import pandas as pd

In [126]:
# Load data
data = pd.read_csv("data/scraped_data.csv")
data.head()

,Price,Address,Title,Fast,Broker,Type,New_building,nFloor,Floor,Height,...,Rooms,Bathrooms,Renovation,Balcony,Body,ID,P_Date,R_Date,Seller_name,HTML_ID
0,"$95,000","Օհանովի փողոց, Երևան",2 սենյականոց բնակարան Օհանովի փողոցում Մալաթիա...,NaN,NaN,Պանելային,Ոչ,9,6,"2,8 մ",...,2,1,Կապիտալ վերանորոգված,Փակ պատշգամբ,Վաճառվում է 2 սենյականոց բնակարան Օհանով փողոց...,22281227,26.03.2025,"29.11.2025,",VAN REALTY,0
1,"$300,000",Պուշկինի փողոց,"2 սենյականոց բնակարան Պուշկինի փողոցում, 58 քմ...",NaN,NaN,Քարե,Ոչ,9,1,"2,8 մ",...,2,1,Դիզայներական ոճով վերանորոգված,Առկա չէ,KN2/787 Կենտրոնում ՝ Պուշկին փողոցում վաճառվո...,22264408,22.03.2025,"27.11.2025,",Աղաջանյաններ ռիելթի ՍՊԸ,1
2,"$87,000","Նորաշեն թաղամաս, Երևան",2 սենյականոց բնակարան նորակառույց շենքում Նորա...,NaN,NaN,Մոնոլիտ,Այո,12,6,3 մ,...,2,1,Կապիտալ վերանորոգված,Բաց պատշգամբ,"Աջափնյակ համայնքի Նորաշեն թաղամասում գտնվող ""Ն...",23115585,14.11.2025,"25.11.2025,",Խաչատրյան անշարժ գույքի գործակալություն,2
3,"$130,000","Սուրբ Գրիգոր Լուսավորչի փողոց, Երևան",1 սենյականոց բնակարան Սուրբ Գրիգոր Լուսավորչի ...,NaN,NaN,Քարե,Ոչ,4,2,3 մ,...,1,1,Մասնակի վերանորոգում,Փակ պատշգամբ,Արշակունյաց-Լուսավորիչ խաչմերուկում 1-2 ձևափոխ...,23026462,21.10.2025,"27.11.2025,",Երկիր Գործակալություն,3
4,"$145,000","Գյուլբենկյան փողոց, Երևան",3 սենյականոց բնակարան Գյուլբենկյան փողոցում Ար...,NaN,NaN,Քարե,Ոչ,5,1,"2,75 մ",...,3,1,Կապիտալ վերանորոգված,Մի քանի պատշգամբ,"Կոմիտաս Գյուլբենկյան 37 Նոր, կապիտալ վերանորոգ...",22113972,14.02.2025,"29.11.2025,",Liana,4


In [127]:
data.columns

Index(['Price', 'Address', 'Title', 'Fast', 'Broker', 'Type', 'New_building',
       'nFloor', 'Floor', 'Height', 'Sqm', 'Rooms', 'Bathrooms', 'Renovation',
       'Balcony', 'Body', 'ID', 'P_Date', 'R_Date', 'Seller_name', 'HTML_ID'],
      dtype='object')

In [128]:
len(data)

23483

In [129]:
data['Price'].unique()

array(['$95,000 ', '$300,000 ', '$87,000 ', ..., '$4,000,000 ',
       '$161,200 ', '$39,500 '], shape=(1069,), dtype=object)

In [130]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23483 entries, 0 to 23482
Data columns (total 21 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Price         23433 non-null  object 
 1   Address       23483 non-null  object 
 2   Title         23483 non-null  object 
 3   Fast          0 non-null      float64
 4   Broker        0 non-null      float64
 5   Type          23483 non-null  object 
 6   New_building  23483 non-null  object 
 7   nFloor        23483 non-null  object 
 8   Floor         23483 non-null  object 
 9   Height        23483 non-null  object 
 10  Sqm           23483 non-null  object 
 11  Rooms         23483 non-null  object 
 12  Bathrooms     23483 non-null  object 
 13  Renovation    23483 non-null  object 
 14  Balcony       23483 non-null  object 
 15  Body          23481 non-null  object 
 16  ID            23483 non-null  int64  
 17  P_Date        23483 non-null  object 
 18  R_Date        22628 non-nu

In [131]:
# Drop unnecessary columns
columns_to_drop = ['Fast', 'Broker', 'Title', 'Body', 'Seller_name', 'R_Date', 'P_Date', 'ID', 'HTML_ID']
data.drop(columns=columns_to_drop, axis=1, inplace=True, errors='ignore')
print(f"Remaining columns: {data.columns.tolist()}.")


Remaining columns: ['Price', 'Address', 'Type', 'New_building', 'nFloor', 'Floor', 'Height', 'Sqm', 'Rooms', 'Bathrooms', 'Renovation', 'Balcony'].


In [132]:
data.columns

Index(['Price', 'Address', 'Type', 'New_building', 'nFloor', 'Floor', 'Height',
       'Sqm', 'Rooms', 'Bathrooms', 'Renovation', 'Balcony'],
      dtype='object')

In [133]:
data["Type"].unique()

array(['Պանելային', 'Քարե', 'Մոնոլիտ', 'Կասետային', 'Աղյուսե', 'Փայտե'],
      dtype=object)

In [134]:
# Remove duplicates
initial_duplicate_count = data.shape[0]
data = data.drop_duplicates(keep='last')
print(f"Removed {initial_duplicate_count - data.shape[0]} duplicates, {data.shape[0]} records remaining.")


Removed 919 duplicates, 22564 records remaining.


In [135]:
def converter(price):
    currency = re.search(r"\$|€|֏|руб|₽|nan", price)[0]
    return price.replace(currency, "").replace(",", ""), currency

# Clean `Price` column
data['Price'].dropna(inplace=True)
data['Price'] = data['Price'].astype(str)

# Separate currency and convert `Price` to numeric
data['Price'], data["Currency"] = zip(*data.Price.apply(converter))
data['Price'] = pd.to_numeric(data['Price'], errors='coerce')
print(f"{data['Price'].count()} valid entries.")


22515 valid entries.


In [136]:
data['Price'].describe()

count    2.251500e+04
mean     1.993021e+06
std      1.295999e+07
min      1.200000e+03
25%      1.000000e+05
50%      1.370000e+05
75%      2.150000e+05
max      6.400000e+08
Name: Price, dtype: float64

In [137]:
data.columns

Index(['Price', 'Address', 'Type', 'New_building', 'nFloor', 'Floor', 'Height',
       'Sqm', 'Rooms', 'Bathrooms', 'Renovation', 'Balcony', 'Currency'],
      dtype='object')

In [138]:
# Map districts from geocoded addresses list
address_districts_map = pd.read_csv("data/all_addresses_with_districts.csv")
data['Address'] = data['Address'].str.strip().str.lower()
data = data.merge(address_districts_map[["Address", "District"]], on='Address', how='left')
data.drop(columns=['Address'], inplace=True)

In [139]:
data['District'].isnull().sum()

np.int64(3329)

In [140]:
data.dropna(subset=['District'], inplace=True)

In [141]:
print(data['Sqm'].isna().sum())
print(data['Sqm'].unique())

0
['68 քմ' '46 քմ' '50 քմ' '70 քմ' '47 քմ' '75 քմ' '36 քմ' '120 քմ' '92 քմ'
 '31 քմ' '98 քմ' '80 քմ' '65 քմ' '200 քմ' '85 քմ' '71 քմ' '94 քմ' '81 քմ'
 '100 քմ' '69 քմ' '93 քմ' '67 քմ' '66 քմ' '63 քմ' '90 քմ' '138 քմ'
 '196 քմ' '78 քմ' '53 քմ' '74 քմ' '76 քմ' '133 քմ' '40 քմ' '95 քմ' '57 քմ'
 '45 քմ' '52 քմ' '87 քմ' '54 քմ' '82 քմ' '60 քմ' '83 քմ' '32 քմ' '18 քմ'
 '72 քմ' '58 քմ' '238 քմ' '134 քմ' '30 քմ' '135 քմ' '84 քմ' '198 քմ'
 '51 քմ' '89 քմ' '37 քմ' '55 քմ' '91 քմ' '96 քմ' '56 քմ' '169 քմ' '125 քմ'
 '113 քմ' '64 քմ' '41 քմ' '142 քմ' '162 քմ' '39 քմ' '110 քմ' '104 քմ'
 '22 քմ' '77 քմ' '143 քմ' '62 քմ' '118 քմ' '114 քմ' '59 քմ' '128 քմ'
 '86 քմ' '44 քմ' '73 քմ' '140 քմ' '38 քմ' '115 քմ' '42 քմ' '105 քմ'
 '106 քմ' '150 քմ' '33 քմ' '97 քմ' '109 քմ' '61 քմ' '203 քմ' '175 քմ'
 '111 քմ' '103 քմ' '88 քմ' '43 քմ' '101 քմ' '236 քմ' '141 քմ' '137 քմ'
 '127 քմ' '306 քմ' '35 քմ' '160 քմ' '48 քմ' '176 քմ' '79 քմ' '107 քմ'
 '182 քմ' '116 քմ' '108 քմ' '112 քմ' '144 քմ' '320 քմ' '117 քմ' '132 քմ'


In [142]:
# Filtering out invalid 'Sqm' values
initial_sqm_count = data.shape[0]
data['Sqm'] = data['Sqm'].astype(str)
data['Sqm'] = data['Sqm'].apply(lambda x: x.replace("քմ", "").strip())
data['Sqm'] = pd.to_numeric(data['Sqm'], errors='coerce')
print(f"{initial_sqm_count} initial records, {data['Sqm'].count()} valid entries after cleaning.")


19235 initial records, 19235 valid entries after cleaning.


In [143]:
print(data['Height'].isna().sum())
print(data['Height'].unique())

0
['2,8 մ' '3 մ' '2,75 մ' '3,5 մ' '2,6 մ' '2,7 մ' '3,2 մ' '2,5 մ']


In [144]:
# Clean and convert 'Height' column
initial_height_count = data['Height'].count()
data['Height'] = data['Height'].astype(str)
data['Height'] = data['Height'].apply(lambda x: x.replace(" մ","").replace(",", ".").strip())
data['Height'] = pd.to_numeric(data['Height'], errors='coerce')
print(f"{initial_height_count} initial records, {data['Height'].count()} valid entries after cleaning.")


19235 initial records, 19235 valid entries after cleaning.


In [145]:
print(data['Rooms'].isna().sum())
print(data['Bathrooms'].isna().sum())


0
0


In [146]:
print(data['Rooms'].unique())
print(data['Bathrooms'].unique())

['2' '1' '3' '4' '5' '6' '7' '8+']
['1' '2' '3+']


In [147]:
# Cleaning 'Rooms' and 'Bathrooms' columns
initial_rooms_count = data['Rooms'].count()
data['Rooms'] = data['Rooms'].replace("8+", "8")
data['Rooms'] = pd.to_numeric(data['Rooms'], errors='coerce')
print(f"{initial_rooms_count} initial records, {data['Rooms'].count()} valid entries after cleaning `Rooms`.")



initial_bathrooms_count = data['Bathrooms'].count()
data['Bathrooms'] = data['Bathrooms'].replace("3+", "3")
data['Bathrooms'] = pd.to_numeric(data['Bathrooms'], errors='coerce')
print(f"{initial_bathrooms_count} initial records, {data['Bathrooms'].count()} valid entries after cleaning `Bathrooms`.")


19235 initial records, 19235 valid entries after cleaning `Rooms`.
19235 initial records, 19235 valid entries after cleaning `Bathrooms`.


In [148]:
print(data['nFloor'].unique())
print(data['Floor'].unique())


['9' '12' '4' '5' '8' '16' '15' '17' '11' '14' '19' '10' '3' '6' '20' '13'
 '2' '7' '18' '22' '21' '1' '23' '25' '27' '24']
['6' '2' '1' '5' '3' '13' '10' '16' '4' '7' '11' '12' '14' '20' '8' '18'
 '9' '15' '17' '19' '21' '22']


In [149]:
print(data['nFloor'].isnull().sum())
print(data['Floor'].isnull().sum())

0
0


In [150]:
# Clean 'Floor' and 'nFloor'
initial_floor_count = data['Floor'].count()
# Since, according to our assumption, 
# 25 is the maximum number of floor that can appear in Yerevan => anything larger is considered as mistake and is replaced
data['Floor'] = data['Floor'].replace("32+", "25")
data['nFloor'] = data['nFloor'].replace("32+", "25")
data['Floor'] = pd.to_numeric(data['Floor'], errors='coerce')
data['nFloor'] = pd.to_numeric(data['nFloor'], errors='coerce')
print(f"{initial_floor_count} initial records, {data['Floor'].count()} valid entries after cleaning.")


19235 initial records, 19235 valid entries after cleaning.


In [151]:
print(f"Final data shape after cleaning: {data.shape}.")

# Save 
data.to_csv("data/cleaned_data.csv", index=False)

Final data shape after cleaning: (19235, 13).
